In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
sys.path.append('/content/drive/MyDrive/ColabNotebooks/EKA_RAG_Project_v2/')

In [4]:
%cd /content/drive/MyDrive/ColabNotebooks/EKA_RAG_Project_v2/

/content/drive/MyDrive/ColabNotebooks/EKA_RAG_Project_v2


In [33]:
!git init
!git config --global user.email "naveenkamal.2011@gmail.com"
!git config --global user.name "Naveen Kamal"
!git pull origin main
!git add .
!git commit -m "Chunk_id conflict"
!git push origin main

In [5]:
!pip install torch transformers sentence-transformers chromadb numpy langfuse python-dotenv # flask

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-http to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.4/475.4 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
from config.config import (LOG_PATH,MAX_GENERATION_LATENCY,MAX_TOTAL_LATENCY,RERANKER_TOP_K,RERANKER_MODEL_NAME,LLM_MODEL_NAME,MAX_NEW_TOKENS,MAX_CONTEXT_TOKENS,TEMPERATURE,DO_SAMPLE,CHUNK_SIZE,CHUNK_OVERLAP,RAW_DOCS_PATH,SIMILARITY_THRESHOLD,CHROMA_DB_PATH,TOP_K,EMBEDDING_MODEL_NAME,REFUSAL_MESSAGE,MAX_TIME)

In [7]:
from models.embedding_model import EmbeddingModel

In [8]:
from models.llm_model import LLMModel

In [9]:
from rag.generator import Generator

In [ ]:
# import sys
# # Uninstall current opentelemetry packages and install compatible versions
# !pip uninstall -y opentelemetry-api opentelemetry-sdk
# !pip install opentelemetry-api==1.12.0 opentelemetry-sdk==1.12.0



In [10]:
from rag.retriever import Retriever

In [16]:
from rag.orchestrator import RAGOrchestrator

In [ ]:
# from ingestion.ingestion import ingest_documents

In [ ]:
# from ingestion.chunker import Chunker

In [17]:
from rag.guard import Guard

In [ ]:
# from models.embedding_model import EmbeddingModel
# from models.llm_model import LLMModel
# from rag.retriever import Retriever
# from rag.generator import Generator
# from rag.orchestrator import RAGOrchestrator
# from ingestion.ingestion import ingest_documents
# # from ingestion.chunker import Chunker
# from rag.guard import Guard

In [14]:
import torch
from transformers import AutoTokenizer,AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import os
import uuid
import importlib
import time
from datetime import datetime

In [18]:
llm_model =None
tokenizer = None
embed_model = None

In [19]:
import chromadb
from chromadb.config  import Settings

In [46]:
import config.config
importlib.reload(config.config)
from config.config import (LOG_PATH,MAX_GENERATION_LATENCY,MAX_TOTAL_LATENCY,RERANKER_TOP_K,RERANKER_MODEL_NAME,LLM_MODEL_NAME,MAX_NEW_TOKENS,MAX_CONTEXT_TOKENS,TEMPERATURE,DO_SAMPLE,CHUNK_SIZE,CHUNK_OVERLAP,RAW_DOCS_PATH,SIMILARITY_THRESHOLD,CHROMA_DB_PATH,TOP_K,EMBEDDING_MODEL_NAME)

In [47]:
import rag.retriever
importlib.reload(rag.retriever)
from rag.retriever import Retriever

In [27]:
import reranker.reranker
importlib.reload(reranker.reranker)
from reranker.reranker import Reranker,select_chunks

In [39]:
import rag.orchestrator
importlib.reload(rag.orchestrator)
from rag.orchestrator import RAGOrchestrator

In [ ]:
import rag.guard
importlib.reload(rag.guard)
from rag.guard import Guard

In [17]:
import config.model_loader
importlib.reload(config.model_loader)
from config.model_loader import get_llm,get_embedder

In [20]:
import observability.logger
importlib.reload(observability.logger)
from observability.logger import Logger

In [38]:
import observability.langfuse_tracer
importlib.reload(observability.langfuse_tracer)
from observability.langfuse_tracer import LangfuseTracer

In [22]:
import observability.anomaly_detector
importlib.reload(observability.anomaly_detector)
from observability.anomaly_detector import AnomalyDetector

In [25]:
from observability.langfuse_tracer import LangfuseTracer

In [22]:
log = """{
    "request_id": "0b5955c0-3294-4718-a5eb-24e9e8f8a66a",
    "timestamp": "2026-04-10T09:01:25.334948",
    "query": "Who is the PM of USA?",
    "answer": "I do not have sufficient information in the knowledge base to answer this question.",
    "confidence": "low",
    "source": null,
    "chunk_ids": [],
    "retrieval_scores": [
      0.16167736053466797,
      0.1574939489364624,
      0.1561795473098755,
      0.1395883560180664,
      0.13692808151245117,
      0.1340998411178589,
      0.13204419612884521
    ],
    "reranker_scores": [
      -11.36478042602539,
      -11.36607551574707,
      -11.37800407409668
    ],
    "rewrite_triggered": false,
    "rewritten_query": null,
    "latency": {
      "total": 3.773,
      "retrieval": 0.004,
      "reranking": 3.734,
      "rewrite": null,
      "generation": 0.0
    },
    "anomalies": [
      "low_confidence",
      "empty_retrieval"
    ]
  }"""

# LangfuseTracer.trace(log)

In [29]:
import json
log = json.loads(log)

In [26]:
client = LangfuseTracer.get_client()

DEBUG:langfuse:Thread: Media upload consumer thread #0 started and actively processing queue items
DEBUG:langfuse:Prompt cache initialized.
DEBUG:langfuse:Startup: Score ingestion consumer thread #0 started with batch size 15 and interval 1s
INFO:langfuse:Startup: Langfuse tracer successfully initialized | public_key=pk-lf-c0563e85-3dd3-4fb2-8785-cb3f5b222e65 | base_url=https://cloud.langfuse.com | environment=default | sample_rate=1.0 | media_threads=1


Public_key=pk-lf-c0563e85-3dd3-4fb2-8785-cb3f5b222e65


In [30]:
LangfuseTracer.trace(log)

Public_key=pk-lf-c0563e85-3dd3-4fb2-8785-cb3f5b222e65


DEBUG:langfuse:Auth check successful, found 1 projects
DEBUG:langfuse:Score: Enqueuing event type=score-create for trace_id=None name=query_size_chars value=21.0
DEBUG:langfuse:Score: Enqueuing event type=score-create for trace_id=None name=latency_total value=3.773
DEBUG:langfuse:Score: Enqueuing event type=score-create for trace_id=None name=latency_retrieval value=0.004
DEBUG:langfuse:Score: Enqueuing event type=score-create for trace_id=None name=latency_reranking value=3.734
ERROR:langfuse:Error creating score: Failed to process score event for trace_id=12674226a168630279cac2d2b34febac, name=latency_rewrite. Error: 2 validation errors for ScoreBody
value.float
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/float_type
value.str
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydant

Flush completed successfully


In [43]:
print(client.auth_check())

DEBUG:langfuse:Auth check successful, found 1 projects


True


In [31]:
print(os.getenv("LANGFUSE_BASE_URL"))

https://cloud.langfuse.com


In [35]:
with client.start_observation(name="eka_request",input={"query":log['query']},output = {"answer":log["answer"]},metadata=log):
  pass
client.flush()

TypeError: 'LangfuseSpan' object does not support the context manager protocol

In [40]:
orch = RAGOrchestrator()

In [ ]:
# llm_model,tokenizer = get_llm()

In [ ]:
import rag.generator
importlib.reload(rag.generator)
from rag.generator import Generator

In [21]:
from models.llm_model import LLMModel

In [22]:
llm = LLMModel()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [23]:
generator = Generator(llm)

In [ ]:
# generator = Generator(llm_model)
# llm_model


In [44]:
# query = "Is Laravel's sample environment configuration  ready to use with Laravel Homestead?"
# query = "By default, Laravel's sample environment configuration is ready to use with Laravel Homestead"
# query = "Where is the database configuration for your application  located?"
# query ="Do each migration file name contains a timestamp?"
# query ="Each migration file name contains a timestamp"
# query = "Predis has been abandoned by the package's original author and may be removed from Laravel in a future release"
# query = "Has Predis been abandoned by the package's original author and may be removed from Laravel in a future release?"
# query ="is there no need to clean strings being passed as bindings?"
# query ="there is no need to clean strings being passed as bindings?"
# query = "you may also paginate Eloquent queries."
# query = "May you  also paginate Eloquent queries?"
# query = "In order to protect you from running seeding commands against your production database"
# query = "what to do,in order to protect you from running seeding commands against your production database?"
# query = "Python is an easy to learn, powerful programming language."
# query = "Is python an easy to learn, powerful programming language?"
# query = "How does laravel handle authentication sessions?"
query = "Who is the PM of USA?"
# query = "How can I configure environment variables after creating a newly created database?" #good high
# query = "How can I configure environment variables to point  a newly created sqlite database?" #good low retrival ok
# query = "Is laravel a good framework?"
# query = "Can you provide me with a step-by-step guide on how to configure my environment variables for a newly created SQLite database?"#good retrival ok high accepted.

# query = "Can you optimize search queries for Elasticsearch using Laravel's Eloquent ORM?"
# query = "After creating a new SQLite database using a command such as touch database/database.sqlite, you can easily configure your environment variables to point to this newly created database by using the database's absolute path: DB_CONNECTION=sqlite"

In [26]:
ret = Retriever()

In [24]:
embedder =get_embedder()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [25]:
embedding_model = EmbeddingModel()

In [27]:
ret.collection.count()

63

In [45]:

# query="What should I do,in order to protect from running seeding commands against production database?" #q006
# query= "How can I configure environment variables to point a newly created sqlite database?"
# query="Which command is used to generate migrations?" #q002

result = orch.run_v1(query)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[-11.36478  -11.378002 -11.423817 -11.41995  -11.36608  -11.457875
 -11.385586]
Score:-11.36478042602539	 distance:0.16167736053466797Chunk:{'text': "results randomly. For example, you may use this method to fetch a random user:\n$randomUser = DB::table('users')\n->inRandomOrder()\n->first();\nreorder\nThe reorder method allows you to remove all the existing orders and optionally apply a new order. For example, you can remove all the existing orders:\n$query = DB::table('users')->orderBy('name');\n$unorderedUsers = $query->reorder()->get();\nTo remove all existing orders and apply a new order, provide the column and direction as arguments to the method:\n$query = DB::table('users')->orderBy('name');\n$usersOrderedByEmail = $query->reorder('email', 'desc')->get();\ngroupBy / having\nThe groupBy and having methods may be used to group the query results. The having method's signature is similar to that of the where method:\n$users = DB::table('users')\n->groupBy('account_id')\n->having('a

DEBUG:langfuse:Auth check successful, found 1 projects
DEBUG:langfuse:Score: Enqueuing event type=score-create for trace_id=None name=query_size_chars value=21.0
DEBUG:langfuse:Score: Enqueuing event type=score-create for trace_id=None name=latency_total value=4.025
DEBUG:langfuse:Score: Enqueuing event type=score-create for trace_id=None name=latency_retrieval value=0.008
DEBUG:langfuse:Score: Enqueuing event type=score-create for trace_id=None name=latency_reranking value=3.991
DEBUG:langfuse:Score: Enqueuing event type=score-create for trace_id=None name=latency_rewrite value=0.0
DEBUG:langfuse:Score: Enqueuing event type=score-create for trace_id=None name=latency_generation value=0.0
DEBUG:langfuse:Score: Enqueuing event type=score-create for trace_id=None name=retrieval_score_0 value=0.16167736053466797
DEBUG:langfuse:Score: Enqueuing event type=score-create for trace_id=None name=retrieval_score_1 value=0.1574939489364624
DEBUG:langfuse:Score: Enqueuing event type=score-create f

Flush completed successfully
[Logger] a8ffd103-efb7-471f-b687-b88eab295339 |latency:4.025s|confidence:low|anomalies:['low_confidence', 'empty_retrieval']


In [55]:
LangfuseTracer.get_client()

To set environment variables securely in Colab, you can use Colab's built-in "Secrets" feature. Click on the 🔑 icon in the left sidebar, then add your `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and `LANGFUSE_BASE_URL` as new secrets. Once added, you can access them in your notebook like this:

In [ ]:
# REFUSAL_MESSAGE = "I do not have sufficient information in the knowledge base to answer this question."
# query ="Is there no need to clean strings being passed as bindings?"#{'total_chunks': 21, 'chunk_id': 0, 'source': 'queries.txt','Similarity':'0.12824028730392456'}

# query = "Which command is used to generate migrations?"#
# query ="Can you provide me with a step-by-step guide on how to configure my environment variables for a newly created SQLite database?"
query = "What should I do,in order to protect from running seeding commands against production database?"

query_embeddings=embedding_model.embed([query])[0]
# results = ret.retrieve(query_embeddings)
results = ret.collection.query(query_embeddings=[query_embeddings], n_results=3, include=["documents", "metadatas", "distances"])
final_result = ({"answer":REFUSAL_MESSAGE,"confidence":"low","source":None,"chunk_id":None})
if(not results or len(results['documents'][0])==0):
  print(final_result)
else:
  chunks = results['documents'][0]



In [ ]:
for document,metadata,distance in zip(results['documents'][0],results['metadatas'][0],results['distances'][0]):
  print(document)
  print(metadata)
  print(f"Similarity={1-distance}")

<s> Database: Seeding
Introduction
Writing Seeders
Using Model Factories
Calling Additional Seeders
Running Seeders
Introduction
Laravel includes a simple method of seeding your database with test data using seed classes. All seed classes are stored in the database/seeds directory. Seed classes may have any name you wish, but probably should follow some sensible convention, such as UserSeeder, etc. By default, a DatabaseSeeder class is defined for you. From this class, you may use the call method to run other seed classes, allowing you to control the seeding order.
Writing Seeders
To generate a seeder, execute the make:seeder Artisan command. All seeders generated by the framework will be placed in the database/seeds directory:
php artisan make:seeder UserSeeder
A seeder class only contains one method by default: run. This method is called when the db:seed Artisan command is executed. Within the run method, you may insert data into your database however you wish. You may use the query 

In [ ]:
 data= {
        "query_id": "q018",
        "query": "Who is the PM of USA?",
        "expected_keywords": [],
        "expected_chunk_id": [],
        "should_be_accepted": false,
        "category": "off-domain",
        "difficulty_level": "Easy"
    }

In [ ]:
from evaluate.metrics import Metrics

In [ ]:
pre=Metrics.retrieval_precision(["2"],["2","3"])
pre

0.5

In [ ]:
rec = Metrics.retrieval_recall(["2"],["2","3"])
rec

1.0

In [ ]:
f1 = Metrics.f1_score(pre,rec)
f1

0.6666666666666666

In [ ]:
print(Metrics.refusal_correct(False,True))

False


In [ ]:
Metrics.keyword_match([
            "Predis",
            "abandoned",
            "yes"
        ],"Yes,Predis is  planned to be removed from Laravel in a future release.")

0.6666666666666666

In [ ]:
!python evaluate.py

[EvalRunner] Running evaluation on 20 queries...

Loading weights: 100% 103/103 [00:00<00:00, 731.84it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100% 201/201 [00:08<00:00, 22.90it/s, Materializing param=model.norm.weight]
Loading weights: 100% 105/105 [00:00<00:00, 2506.97it/s, Materializing param=classifier.weight]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arc

In [ ]:
# %%writefile /content/drive/MyDrive/ColabNotebooks/EKA_RAG_Project_v2/evaluate/report.py
# import json
# import os

# class Report:
#     @staticmethod
#     def generate(results, output_path="evaluate/report.json"):
#         # Ensure the directory for the output file exists
#         output_dir = os.path.dirname(output_path)
#         if output_dir: # Only proceed if output_path includes a directory
#             # Use a Python 3.1 or earlier compatible way to create directories
#             # as exists_ok might not be supported in the environment.
#             if not os.path.exists(output_dir):
#                 os.makedirs(output_dir)

#         # Placeholder: This part of the code would normally process 'results'
#         # to calculate various metrics and structure the report.
#         # Ensure 'avg_latency' is calculated and included in the 'summary' here.
#         # For now, we'll create a dummy report structure for demonstration.
#         report = {
#             "summary": {
#                 "total_queries": len(results),
#                 # Add other summary metrics as computed
#                 # For instance, if you compute avg_latency elsewhere, ensure it's added:
#                 # "avg_latency": calculate_average_latency(results)
#             },
#             "details": results # Assuming 'results' contains detailed evaluation outcomes
#         }

#         with open(output_path, 'w') as f:
#             json.dump(report, f, indent=4)
#         Report._print_summary(report)

#     @staticmethod
#     def _print_summary(report):
#         s = report.get('summary', {}) # Safely get summary dictionary
#         print("--- Evaluation Summary ---")
#         print(f"Total Queries: {s.get('total_queries', 'N/A')}")
#         # Safely access 'avg_latency' with a default value if not present
#         print(f"Avg Latency: {s.get('avg_latency', 'N/A')}s")
#         # Add other summary metrics to print as needed
#         print("--------------------------")


[X] q001 | F1 : 0.00 | KW: 0.20 | Refusal:False | Latency : 17.707
[Success] q002 | F1 : 0.00 | KW: 0.00 | Refusal:True | Latency : 127.879
[Success] q008 | F1 : 0.00 | KW: 0.00 | Refusal:True | Latency : 4.038
==================================================
EVALUATION REPORT
==================================================
Total Queries : 3
Precision: 0.0
Recall: 0.0
F1: 0.0
Keyword Match: 0.1
Avg Latency: 50s
Refusal Accuracy: 1
Rewrite Trigger: 0
Rewrite Sucess %: 0.0
==================================================

[Report] Saved to evaluate/report.json

**************************************************
[{'id': 'q001', 'query': 'Is there no need to clean strings being passed as bindings?', 'category': 'domain', 'difficulty_level': 'Easy', 'expected_chunk': ['0'], 'retrieved_ids': [], 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'keyword_match': 0.2, 'refusal_correct': False, 'should_refuse': False, 'was_refused': True, 'confidence': 'low', 'rewrite_triggered': False, 'latency_seconds': 17.707, 'answer': 'I do not have sufficient information in the knowledge base to answer this question.'}, {'id': 'q002', 'query': 'Which command is used to generate migrations?', 'category': 'domain', 'difficulty_level': 'Easy', 'expected_chunk': ['0'], 'retrieved_ids': [], 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'keyword_match': 0.0, 'refusal_correct': True, 'should_refuse': False, 'was_refused': False, 'confidence': 'HIGH', 'rewrite_triggered': False, 'latency_seconds': 127.879, 'answer': "### System :\n\t\t\t\t   Generate Migrations\n\t\t\t\t   ### Rules:\n\t\t\t\t   1. Provide the context to the question.\n\t\t\t\t   2. Use the provided context to answer the question.\n\t\t\t\t   3. Cite the chunk id when answering.\n\t\t\t\t   4. Do not repeat instructions.\n\t\t\t\t   5. Do not use prior knowledge.\n\t\t\t\t   6. Do not repeat Knowledge_Base section.\n\t\t\t\t   7. Answer concisely.\n\t\t\t\t   8. Only output the final answer.\n\n\t\t\t\t   ### Knowledge_Base:\n\t\t\t\t   Context 1:\n\t\t\t\t   Migrations are like version control for your database, allowing your team to modify and share the application's database schema. Migr"}, {'id': 'q008', 'query': 'What is the core reason behind USA, Iran war?', 'category': 'off-domain', 'difficulty_level': 'Easy', 'expected_chunk': [], 'retrieved_ids': [], 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'keyword_match': 0.0, 'refusal_correct': True, 'should_refuse': True, 'was_refused': True, 'confidence': 'low', 'rewrite_triggered': False, 'latency_seconds': 4.038, 'answer': 'I do not have sufficient information in the knowledge base to answer this question.'}]
**************************************************

{
  "timestamp": "2026-03-31T16:12:57.029174",
  "total_queries": 1,
  "summary": {
    "precision": 0.0,
    "recall": 0.0,
    "f1": 0.0,
    "keyword_match": 0.2,
    "avg_latency_second": 24,
    "refusal_accuracy": 0,
    "rewrite_trigger_rate": 0,
    "rewrite_success_rate": 0.0
  },
  "by_difficulty": {
    "Easy": {
      "count": 1,
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0
    }
  },
  "by_category": {
    "domain": {
      "count": 1,
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0
    }
  },
  "failures": [
    {
      "id": "q001",
      "query": "Is there no need to clean strings being passed as bindings?",
      "category": "domain",
      "difficulty_level": "Easy",
      "expected_chunk": [
        "0"
      ],
      "retrieved_ids": [],
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0,
      "keyword_match": 0.2,
      "refusal_correct": false,
      "should_refuse": false,
      "was_refused": true,
      "confidence": "low",
      "rewrite_triggered": false,
      "latency_seconds": 23.756,
      "answer": "I do not have sufficient information in the knowledge base to answer this question."
    }
  ],
  "raw_results": [
    {
      "id": "q001",
      "query": "Is there no need to clean strings being passed as bindings?",
      "category": "domain",
      "difficulty_level": "Easy",
      "expected_chunk": [
        "0"
      ],
      "retrieved_ids": [],
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0,
      "keyword_match": 0.2,
      "refusal_correct": false,
      "should_refuse": false,
      "was_refused": true,
      "confidence": "low",
      "rewrite_triggered": false,
      "latency_seconds": 23.756,
      "answer": "I do not have sufficient information in the knowledge base to answer this question."
    }
  ]
}

You can set your Hugging Face token as an environment variable. This allows the `transformers` library to use it for authentication when loading models, especially for models that require authentication or to bypass rate limits.

In [28]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

print("HF_TOKEN environment variable set from Colab secrets. This token will be used for Hugging Face model requests.")

HF_TOKEN environment variable set from Colab secrets. This token will be used for Hugging Face model requests.


After setting the `HF_TOKEN`, when you load models using `AutoTokenizer` or `AutoModelForCausalLM`, the library will automatically use this token for authentication.

In [ ]:
# {
#   "timestamp": "2026-03-31T16:12:57.029174",
#   "total_queries": 1,
#   "summary": {
#     "precision": 0.0,
#     "recall": 0.0,
#     "f1": 0.0,
#     "keyword_match": 0.2,
#     "avg_latency_second": 24,
#     "refusal_accuracy": 0,
#     "rewrite_trigger_rate": 0,
#     "rewrite_success_rate": 0.0
#   },
#   "by_difficulty": {
#     "Easy": {
#       "count": 1,
#       "precision": 0.0,
#       "recall": 0.0,
#       "f1": 0.0
#     }
#   },
#   "by_category": {
#     "domain": {
#       "count": 1,
#       "precision": 0.0,
#       "recall": 0.0,
#       "f1": 0.0
#     }
#   },
#   "failures": [
#     {
#       "id": "q001",
#       "query": "Is there no need to clean strings being passed as bindings?",
#       "category": "domain",
#       "difficulty_level": "Easy",
#       "expected_chunk": [
#         "0"
#       ],
#       "retrieved_ids": [],
#       "precision": 0.0,
#       "recall": 0.0,
#       "f1": 0.0,
#       "keyword_match": 0.2,
#       "refusal_correct": false,
#       "should_refuse": false,
#       "was_refused": true,
#       "confidence": "low",
#       "rewrite_triggered": false,
#       "latency_seconds": 23.756,
#       "answer": "I do not have sufficient information in the knowledge base to answer this question."
#     }
#   ],
#   "raw_results": [
#     {
#       "id": "q001",
#       "query": "Is there no need to clean strings being passed as bindings?",
#       "category": "domain",
#       "difficulty_level": "Easy",
#       "expected_chunk": [
#         "0"
#       ],
#       "retrieved_ids": [],
#       "precision": 0.0,
#       "recall": 0.0,
#       "f1": 0.0,
#       "keyword_match": 0.2,
#       "refusal_correct": false,
#       "should_refuse": false,
#       "was_refused": true,
#       "confidence": "low",
#       "rewrite_triggered": false,
#       "latency_seconds": 23.756,
#       "answer": "I do not have sufficient information in the knowledge base to answer this question."
#     }
#   ]
# }

In [ ]:
!ls evaluate

dataset1.json  eval_runner.py  metrics.py   report.json
dataset.json   __init__.py     __pycache__  report.py


In [ ]:
# %%writefile /content/drive/MyDrive/ColabNotebooks/EKA_RAG_Project_v2/evaluate/report.py
# import json

# class Report:
#     @staticmethod
#     def generate(results, output_path="evaluate/report.json"):
#         # Placeholder: This part of the code would normally process 'results'
#         # to calculate various metrics and structure the report.
#         # Ensure 'avg_latency' is calculated and included in the 'summary' here.
#         # For now, we'll create a dummy report structure for demonstration.
#         report = {
#             "summary": {
#                 "total_queries": len(results),
#                 # Add other summary metrics as computed
#                 # For instance, if you compute avg_latency elsewhere, ensure it's added:
#                 # "avg_latency": calculate_average_latency(results)
#             },
#             "details": results # Assuming 'results' contains detailed evaluation outcomes
#         }

#         with open(output_path, 'w') as f:
#             json.dump(report, f, indent=4)
#         Report._print_summary(report)

#     @staticmethod
#     def _print_summary(report):
#         s = report.get('summary', {}) # Safely get summary dictionary
#         print("--- Evaluation Summary ---")
#         print(f"Total Queries: {s.get('total_queries', 'N/A')}")
#         # Safely access 'avg_latency' with a default value if not present
#         print(f"Avg Latency: {s.get('avg_latency', 'N/A')}s")
#         # Add other summary metrics to print as needed
#         print("--------------------------")

The `evaluate/report.py` file has been updated with a more robust way to handle potentially missing keys.

Please run the `evaluate.py` script again to see if the error is resolved:

```python
!python evaluate.py
```

In [ ]:
questions=[
               {
			   'query_id':'q001',
			   'query':'Is there no need to clean strings being passed as bindings?',
			   'expected_keywords':["yes","no","need","clean","strings"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q002',
			   'query':'Which command is used to generate migrations?',
			   'expected_keywords':["command","make:migration","artisan"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q003',
			   'query':'How can I use seeders?',
			   'expected_keywords':["generate","seeder","execute","make:seeder","artisan"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q004',
			   'query':'Explain redis facade with example?',
			   'expected_keywords':["Redis","Facade","Redis GET command","use Illuminate\\Support\\Facades\\Redis;"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Hard'
			   },
			   {
			   'query_id':'q005',
			   'query':'What is redis and how it could be installed?',
			   'expected_keywords':["Redis","data structure","install","PhpRedis","PECL","predis","composer"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Hard'
			   },
			   {
			   'query_id':'q006',
			   'query':'What should I do,in order to protect from running seeding commands against production database?',
			   'expected_keywords':["protect","running","seeding","commands","production","prompted","confirmation"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Medium'
			   },
			   {
			   'query_id':'q007',
			   'query':'Which country won the T20 cricket world cup 2026? ',
			   'expected_keywords':[],
			   'expected_chunk_id':[],
			   'should_be_accepted':False,
			   'category':'off-domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q008',
			   'query':'What is the core reason behind USA, Iran war?',
			   'expected_keywords':[],
			   'expected_chunk_id':[],
			   'should_be_accepted':False,
			   'category':'off-domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q009',
			   'query':'How can I configure environment variables to point a newly created sqlite database?',
			   'expected_keywords':["DB_CONNECTION","sqlite","DB_DATABASE","database.sqlite"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
               {
			   'query_id':'q010',
			   'query':'why is there no need to clean strings being passed as bindings?',
			   'expected_keywords':["Laravel","query","builder","uses","PDO","parameter","binding"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'edge',
			   'difficulty_level': 'Hard'
			   },
			   {
			   'query_id':'q011',
			   'query':'What is the population of republic of china?',
			   'expected_keywords':[],
			   'expected_chunk_id':[],
			   'should_be_accepted':False,
			   'category':'off-domain',
			   'difficulty_level': 'Easy'
			   },
               {
			   'query_id':'q012',
			   'query':'Has Predis been abandoned by the package\'s original author and may be removed from Laravel in a future release?',
			   'expected_keywords':["Predis","abandoned","yes"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q013',
			   'query':'Has Predis been abandoned and likely to be removed?',
			   'expected_keywords':["Predis","abandoned","yes"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Medium'
			   },
			   {
			   'query_id':'q014',
			   'query':'Do each migration file name contains a timestamp?',
			   'expected_keywords':["yes","migration","file","contains","timestamp"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q015',
			   'query':'Where is the database configuration for your application located?',
			   'expected_keywords':["database","configuration","located","config","database.php","config/database.php"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q016',
			   'query':'Is Laravel\'s sample environment configuration ready to use with Laravel Homestead?',
			   'expected_keywords':["By","default","Laravel","configuration","Homestead"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q017',
			   'query':'Can you provide me with a step-by-step guide on how to configure my environment variables for a newly created SQLite database?',
			   'expected_keywords':["DB_CONNECTION","sqlite","DB_DATABASE","database.sqlite"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Hard'
			   },
			   {
			   'query_id':'q018',
			   'query':'Who is the PM of USA?',
			   'expected_keywords':[],
			   'expected_chunk_id':[],
			   'should_be_accepted':False,
			   'category':'off-domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q019',
			   'query':'Is python an easy to learn, powerful programming language?',
			   'expected_keywords':[],
			   'expected_chunk_id':[],
			   'should_be_accepted':False,
			   'category':'off-domain',
			   'difficulty_level': 'Medium'
			   },
			   {
			   'query_id':'q020',
			   'query':'How does laravel handle authentication sessions?', #As authentication.txt not included in ingestion
			   'expected_keywords':[],
			   'expected_chunk_id':[],
			   'should_be_accepted':False,
			   'category':'off-domain',
			   'difficulty_level': 'Medium'
			   }

]

In [ ]:
import json

data = [
               {
			   'query_id':'q001',
			   'query':'Is there no need to clean strings being passed as bindings?',
			   'expected_keywords':["yes","no","need","clean","strings"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q002',
			   'query':'Which command is used to generate migrations?',
			   'expected_keywords':["command","make:migration","artisan"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q003',
			   'query':'How can I use seeders?',
			   'expected_keywords':["generate","seeder","execute","make:seeder","artisan"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q004',
			   'query':'Explain redis facade with example?',
			   'expected_keywords':["Redis","Facade","Redis GET command","use Illuminate\\Support\\Facades\\Redis;"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Hard'
			   },
			   {
			   'query_id':'q005',
			   'query':'What is redis and how it could be installed?',
			   'expected_keywords':["Redis","data structure","install","PhpRedis","PECL","predis","composer"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Hard'
			   },
			   {
			   'query_id':'q006',
			   'query':'What should I do,in order to protect from running seeding commands against production database?',
			   'expected_keywords':["protect","running","seeding","commands","production","prompted","confirmation"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Medium'
			   },
			   {
			   'query_id':'q007',
			   'query':'Which country won the T20 cricket world cup 2026? ',
			   'expected_keywords':[],
			   'expected_chunk_id':[],
			   'should_be_accepted':False,
			   'category':'off-domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q008',
			   'query':'What is the core reason behind USA, Iran war?',
			   'expected_keywords':[],
			   'expected_chunk_id':[],
			   'should_be_accepted':False,
			   'category':'off-domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q009',
			   'query':'How can I configure environment variables to point a newly created sqlite database?',
			   'expected_keywords':["DB_CONNECTION","sqlite","DB_DATABASE","database.sqlite"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
               {
			   'query_id':'q010',
			   'query':'why is there no need to clean strings being passed as bindings?',
			   'expected_keywords':["Laravel","query","builder","uses","PDO","parameter","binding"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'edge',
			   'difficulty_level': 'Hard'
			   },
			   {
			   'query_id':'q011',
			   'query':'What is the population of republic of china?',
			   'expected_keywords':[],
			   'expected_chunk_id':[],
			   'should_be_accepted':False,
			   'category':'off-domain',
			   'difficulty_level': 'Easy'
			   },
               {
			   'query_id':'q012',
			   'query':'Has Predis been abandoned by the package\'s original author and may be removed from Laravel in a future release?',
			   'expected_keywords':["Predis","abandoned","yes"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q013',
			   'query':'Has Predis been abandoned and likely to be removed?',
			   'expected_keywords':["Predis","abandoned","yes"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Medium'
			   },
			   {
			   'query_id':'q014',
			   'query':'Do each migration file name contains a timestamp?',
			   'expected_keywords':["yes","migration","file","contains","timestamp"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q015',
			   'query':'Where is the database configuration for your application located?',
			   'expected_keywords':["database","configuration","located","config","database.php","config/database.php"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q016',
			   'query':'Is Laravel\'s sample environment configuration ready to use with Laravel Homestead?',
			   'expected_keywords':["By","default","Laravel","configuration","Homestead"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q017',
			   'query':'Can you provide me with a step-by-step guide on how to configure my environment variables for a newly created SQLite database?',
			   'expected_keywords':["DB_CONNECTION","sqlite","DB_DATABASE","database.sqlite"],
			   'expected_chunk_id':[],
			   'should_be_accepted':True,
			   'category':'domain',
			   'difficulty_level': 'Hard'
			   },
			   {
			   'query_id':'q018',
			   'query':'Who is the PM of USA?',
			   'expected_keywords':[],
			   'expected_chunk_id':[],
			   'should_be_accepted':False,
			   'category':'off-domain',
			   'difficulty_level': 'Easy'
			   },
			   {
			   'query_id':'q019',
			   'query':'Is python an easy to learn, powerful programming language?',
			   'expected_keywords':[],
			   'expected_chunk_id':[],
			   'should_be_accepted':False,
			   'category':'off-domain',
			   'difficulty_level': 'Medium'
			   },
			   {
			   'query_id':'q020',
			   'query':'How does laravel handle authentication sessions?', #As authentication.txt not included in ingestion
			   'expected_keywords':[],
			   'expected_chunk_id':[],
			   'should_be_accepted':False,
			   'category':'off-domain',
			   'difficulty_level': 'Medium'
			   }

]

json_output = json.dumps(data, indent=4)
print(json_output)

[
    {
        "query_id": "q001",
        "query": "Is there no need to clean strings being passed as bindings?",
        "expected_keywords": [
            "yes",
            "no",
            "need",
            "clean",
            "strings"
        ],
        "expected_chunk_id": [],
        "should_be_accepted": true,
        "category": "domain",
        "difficulty_level": "Easy"
    },
    {
        "query_id": "q002",
        "query": "Which command is used to generate migrations?",
        "expected_keywords": [
            "command",
            "make:migration",
            "artisan"
        ],
        "expected_chunk_id": [],
        "should_be_accepted": true,
        "category": "domain",
        "difficulty_level": "Easy"
    },
    {
        "query_id": "q003",
        "query": "How can I use seeders?",
        "expected_keywords": [
            "generate",
            "seeder",
            "execute",
            "make:seeder",
            "artisan"
        ],
      

In [ ]:
with open('evaluate/dataset.json', 'w') as f:
    f.write(json_output)
print('JSON data saved to dataset.json')

JSON data saved to queries.json


In [ ]:
questions[0]

{'query_id': 'q001',
 'query': 'Is there no need to clean strings being passed as bindings?',
 'expected_keywords': ['yes', 'no', 'need', 'clean', 'strings'],
 'expected_chunk_id': [],
 'should_be_accepted': True,
 'category': 'domain',
 'difficulty_level': 'Easy'}

In [29]:
query = "What should I do,in order to protect from running seeding commands against production database?"
result = orch.run_v1(query)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

[  0.40496576   4.0457172   -4.234503   -10.648914   -10.574446
  -7.4879274    5.429908  ]
Score:0.40496575832366943	 distance:0.5530707836151123Chunk:{'text': "<s> Database: Seeding\nIntroduction\nWriting Seeders\nUsing Model Factories\nCalling Additional Seeders\nRunning Seeders\nIntroduction\nLaravel includes a simple method of seeding your database with test data using seed classes. All seed classes are stored in the database/seeds directory. Seed classes may have any name you wish, but probably should follow some sensible convention, such as UserSeeder, etc. By default, a DatabaseSeeder class is defined for you. From this class, you may use the call method to run other seed classes, allowing you to control the seeding order.\nWriting Seeders\nTo generate a seeder, execute the make:seeder Artisan command. All seeders generated by the framework will be placed in the database/seeds directory:\nphp artisan make:seeder UserSeeder\nA seeder class only contains one method by default: ru

KeyError: 'reranked_score'

In [ ]:
"""1   similarity': 0.4261389970779419     Score:6.3368425369262695    1
2   similarity': 0.4191470146179199     Score:-10.935098648071289   6
3.  'similarity': 0.4134495258331299     Score:-3.3252553939819336  3
4.   'similarity': 0.41149914264678955   Score:-2.345608711242676   2
5.  'similarity': 0.39291030168533325    Score:-11.421125411987305   7
6.  'similarity': 0.3928818702697754     Score:-6.119968891143799    5
7. 'similarity': 0.3787959814071655      Score:-5.993478775024414    4"""


"1   similarity': 0.4261389970779419     Score:6.3368425369262695    1\n2   similarity': 0.4191470146179199     Score:-10.935098648071289   6\n3.  'similarity': 0.4134495258331299     Score:-3.3252553939819336  3\n4.   'similarity': 0.41149914264678955   Score:-2.345608711242676   2\n5.  'similarity': 0.39291030168533325    Score:-11.421125411987305   7\n6.  'similarity': 0.3928818702697754     Score:-6.119968891143799    5\n7. 'similarity': 0.3787959814071655      Score:-5.993478775024414    4"

In [ ]:
# query = "By default, Laravel's sample environment configuration is ready to use with Laravel Homestead"
0.4261389970779419-0.41149914264678955    #  Reranked gap

0.014639854431152344

In [ ]:
0.4261389970779419-0.4191470146179199     #  Similarity  .42 gap   .05

0.006991982460021973

In [ ]:
print(result['answer'])
print(result['confidence'])
print(result['source'])
print(result['chunk_id'])

I do not have sufficient information in the knowledge base to answer this question.
low
None
None


In [ ]:
from sentence_transformers import CrossEncoder
from reranker.reranker import Reranker

query ="how to implement authentication in Laravel."
test_chunks = [
{'text':'Laravel provides a simple way to organize authorization logic...','similarity':0.45,'metadata':{'source':'doc1','chunk_id':1}},
{'text':'Authentication in laravel is handled via guards and providers...','similarity':0.38,'metadata':{'source':'doc2','chunk_id':2}},
{'text':'The artisan command line tool helps scaffold controllers...','similarity':0.41,'metadata':{'source':'doc3','chunk_id':3}},
]


In [ ]:
import reranker.reranker
importlib.reload(reranker.reranker)
from reranker.reranker import Reranker
reranker = Reranker()


In [ ]:
reranked = reranker.rerank(query=query,chunks=test_chunks,top_n=2)
for i,chunk in enumerate(reranked):
  print(f"Rank {i+1} | reranker_score:{chunk['reranker_score']:.4f}")
  print(f"		text:{chunk['text'][:60]}")
  print()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[  0.5642123   4.142338  -11.366939 ]
Score:0.5642123222351074	 Chunk:{'text': 'Laravel provides a simple way to organize authorization logic...', 'similarity': 0.45, 'metadata': {'source': 'doc1', 'chunk_id': 1}}
Score:4.142337799072266	 Chunk:{'text': 'Authentication in laravel is handled via guards and providers...', 'similarity': 0.38, 'metadata': {'source': 'doc2', 'chunk_id': 2}}
Score:-11.366938591003418	 Chunk:{'text': 'The artisan command line tool helps scaffold controllers...', 'similarity': 0.41, 'metadata': {'source': 'doc3', 'chunk_id': 3}}
Rank 1 | reranker_score:4.1423
		text:Authentication in laravel is handled via guards and provider

Rank 2 | reranker_score:0.5642
		text:Laravel provides a simple way to organize authorization logi



In [ ]:
# %%writefile /content/drive/MyDrive/Colab Notebooks/EKA_RAG_Project_v2/reranker/reranker.py
from sentence_transformers import CrossEncoder
from config.config import RERANKER_MODEL_NAME, RERANKER_TOP_K

class Reranker:
    def __init__(self):
        self.cross_encoder = CrossEncoder(RERANKER_MODEL_NAME)
        self.top_k = RERANKER_TOP_K

    def rerank(self, query: str, chunks: list, top_n: int = None) -> list:
        if top_n is None:
            top_n = self.top_k

        # Prepare sentence pairs for the cross-encoder
        sentence_pairs = [[query, chunk['text']] for chunk in chunks]

        # Get scores from the cross-encoder
        scores = self.cross_encoder.predict(sentence_pairs)

        # Pair scores with original chunks and sort
        scored_chunks = []
        for i, chunk in enumerate(chunks):
            chunk_copy = chunk.copy()  # Avoid modifying original dict
            chunk_copy['reranker_score'] = float(scores[i]) # Ensure score is a float
            scored_chunks.append(chunk_copy)

        scored_chunks.sort(key=lambda x: x['reranker_score'], reverse=True)

        return scored_chunks[:top_n]

The `reranker/reranker.py` file has been updated. Please run the following cells in order to apply the changes and re-run the reranking:

1.  **Cell `adw7cWYAvKa3`**: `from config.config import (RERANKER_TOP_K,RERANKER_MODEL_NAME,LLM_MODEL_NAME,MAX_NEW_TOKENS,MAX_CONTEXT_TOKENS,TEMPERATURE,DO_SAMPLE,CHUNK_SIZE,CHUNK_OVERLAP,RAW_DOCS_PATH,SIMILARITY_THRESHOLD,CHROMA_DB_PATH,TOP_K,EMBEDDING_MODEL_NAME)` (to ensure `RERANKER_MODEL_NAME` and `RERANKER_TOP_K` are loaded)
2.  **Cell `F8RjC8qtYQKb`**: `from reranker.reranker import Reranker` (to reload the updated class definition)
3.  **Cell `G419Uz5EZjmz`**: `reranker = Reranker()` (to re-instantiate the `Reranker` object with the new definition)
4.  **Cell `JT62tXTAZqpf`**: `reranked = reranker.rerank(query=query,chunks=test_chunks,top_n=2)` (to execute the reranking)

In [ ]:
import rewriter.query_rewriter
importlib.reload(rewriter.query_rewriter)
from rewriter.query_rewriter import QueryRewriter

In [ ]:
qrewritten = QueryRewriter(llm)

In [ ]:
print(qrewritten)
print(qrewritten.llm)
print(qrewritten.REWRITE_TEMPLATE)


      ### System:
      You are a search query optimizer.

      ### Rules:
      - Rephrase the question using formal, document-style language.
      - Use technical vocabulary that would appear in documentation.
      - Do not add new information or assumptions.
      - Do not answer the question.
      - Output only the rephrased question. Nothing else.
      
      ### Original Question:
      {query}

      ### Rephrased Question:   
    
    


In [ ]:
query = "After creating a new SQLite database using a command such as touch database/database.sqlite, you can easily configure your environment variables to point to this newly created database by using the database's absolute path: DB_CONNECTION=sqlite"
query_text = query # Use the 'query' variable which holds the statement
rewritten_query = qrewritten.rewrite(query_text)
print(f"Original Statement: {query_text}")
print(f"Rewritten Question: {rewritten_query}")

Original Statement: After creating a new SQLite database using a command such as touch database/database.sqlite, you can easily configure your environment variables to point to this newly created database by using the database's absolute path: DB_CONNECTION=sqlite
Rewritten Question: Can you provide me with a step-by-step guide on how to configure my environment variables for a newly created SQLite database?


In [ ]:
qrewritten.REWRITE_TEMPLATE.format(query=query)

'\n      ### System:\n      You are a search query optimizer.\n\n      ### Rules:\n      - Rephrase the question using formal, document-style language.\n      - Use technical vocabulary that would appear in documentation.\n      - Do not add new information or assumptions.\n      - Do not answer the question.\n      - Output only the rephrased question. Nothing else.\n      \n      ### Original Question:\n      May you  also paginate Eloquent queries?\n\n      ### Rephrased Question:   \n    \n    '

In [ ]:
rewritten_query = qrewritten.rewrite(query)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
rewritten_query

"Can you optimize search queries for Elasticsearch using Laravel's Eloquent ORM?"

 ### Original Question:May you  also paginate Eloquent queries?
 ### Rephrased Question:Can you optimize search queries for Elasticsearch using Laravel's Eloquent ORM?

In [53]:
import os
from langfuse import Langfuse

class LangfuseTracer:
    _client = None

    @classmethod
    def get_client(cls):
        if cls._client is None:
            cls._client = Langfuse(
                public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
                secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
                host=os.getenv("LANGFUSE_BASE_URL"),
            )
        return cls._client

    @staticmethod
    def trace(log: dict):
        client = LangfuseTracer.get_client()
        if client is None:
            print("Langfuse client not initialized. Skipping trace.")
            return

        # Extract chunk_ids to be used as tags for filterability
        # Ensure each chunk ID is a string if it's not already.
        chunk_ids_as_tags = [str(cid) for cid in log.get('chunk_ids', [])]

        # Create a new metadata dictionary, excluding chunk_ids to prevent redundancy
        # if they are also being passed as tags. Langfuse can handle both, but this is cleaner.
        metadata_for_trace = {k: v for k, v in log.items() if k != 'chunk_ids'}

        with client.start_trace(
            name="eka_request",
            input={"query": log['query']},
            output={"answer": log["answer"]},
            metadata=metadata_for_trace, # Pass other log data as metadata
            tags=chunk_ids_as_tags      # Pass chunk_ids as tags for filterability
        ) as trace_obj:
            # No further operations needed here, the trace is created and will be ended automatically
            pass

ImportError: cannot import name 'trace' from 'langfuse' (/usr/local/lib/python3.12/dist-packages/langfuse/__init__.py)

import importlib
import observability.logger
importlib.reload(observability.logger)
import observability.langfuse_tracer # Ensure langfuse_tracer is also reloaded
importlib.reload(observability.langfuse_tracer)
from observability.logger import Logger, logger

import rag.orchestrator
importlib.reload(rag.orchestrator)
from rag.orchestrator import RAGOrchestrator

orch = RAGOrchestrator()
print("Modules reloaded and RAGOrchestrator re-instantiated.")

In [ ]:
from flask import Flask,render_template
import threading
import time

app= Flask(__name__)

@app.route("/",methods=["GET"])
def index():
    return render_template('index.html')

@app.route("/test")
def test():
    return "Hello from Flask test route!"

def run_flask_app():
    # Use debug=True and a specific port. Colab's built-in tunneling will often detect this.
    # use_reloader=False is crucial when running Flask in a separate thread to prevent errors.
    app.run(debug=True, port=5002, use_reloader=False)

if __name__ == "__main__":
    # Start Flask app in a separate thread to avoid blocking the Colab cell
    flask_thread = threading.Thread(target=run_flask_app)
    flask_thread.daemon = True # Allows the main program to exit even if this thread is still running
    flask_thread.start()

    # Give Flask a moment to start up
    time.sleep(3)

    print("Flask app is running in the background on port 5002.")
    print("Now setting up ngrok tunnel...")

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5002
INFO:werkzeug:Press CTRL+C to quit


Flask app is running in the background on port 5002.
Now setting up ngrok tunnel...


In [ ]:
!pwd

/content/drive/MyDrive/Colab Notebooks/EKA_RAG_Project_v2


### Alternative: Using `localtunnel`

If Colab's built-in port forwarding isn't working for you, `localtunnel` is a great alternative. First, we need to install it:

In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 3s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

Now, you can run `localtunnel` to expose your Flask app running on port 5000. This will provide a public URL for your application.

In [ ]:
# get_ipython().system_raw('killall localtunnel || true && npx localtunnel --port 5000 --skip-check &')
# Commenting out localtunnel as it's causing issues.

In [ ]:
# import time
# time.sleep(5)
# !cat url.txt
# Commenting out localtunnel related command

In [ ]:
from ngrok import ngrok

# IMPORTANT: Set your ngrok authentication token here
# You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# Replace 'YOUR_AUTH_TOKEN_HERE' with your actual token.
ngrok.set_auth_token('YOUR_AUTH_TOKEN_HERE')

In [ ]:
# !python app.py & # Flask app is now run directly in the notebook via threading

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
Press CTRL+C to quit


In [ ]:
tunnel = ngrok.connect(5002)
public_url = tunnel.public_url
print(f"ngrok tunnel is active at: {public_url}")

<Task pending name='Task-6' coro=<wrap() running at ngrok_wrapper:6>>


In [ ]:
filtered_chunks = [{'text': "<s> Database: Getting Started\nIntroduction\nConfiguration\nRead & Write Connections\nUsing Multiple Database Connections\nRunning Raw SQL Queries\nListening For Query Events\nDatabase Transactions\nIntroduction\nLaravel makes interacting with databases extremely simple across a variety of database backends using either raw SQL, the fluent query builder, and the Eloquent ORM. Currently, Laravel supports four databases:\nMySQL 5.6+ (Version Policy)\nPostgreSQL 9.4+ (Version Policy)\nSQLite 3.8.8+\nSQL Server 2017+ (Version Policy)\nConfiguration\nThe database configuration for your application is located at config/database.php. In this file you may define all of your database connections, as well as specify which connection should be used by default. Examples for most of the supported database systems are provided in this file.\nBy default, Laravel's sample environment configuration is ready to use with Laravel Homestead, which is a convenient virtual machine for doing Laravel development on your local machine. You are free to modify this configuration as needed for your local database.\nSQLite Configuration\nAfter creating a new SQLite database using a command such as touch database/database.sqlite, you can easily configure your environment variables to point to this newly created database by using the database's absolute path:\nDB_CONNECTION=sqlite\nDB_DATABASE=/absolute/path/to/database.sqlite\nTo enable foreign key constraints for SQLite connections, you should set the DB_FOREIGN_KEYS environment variable to true:\nDB_FOREIGN_KEYS=true\nConfiguration Using URLs\nTypically, database connections are configured using multiple configuration values such as host, database, username, password, etc. Each of these configuration values has its own corresponding environment variable. This means that when configuring your database connection information on a production server", 'similarity': 0.6476882696151733, 'metadata': {'chunk_id': 0, 'source': 'database.txt', 'total_chunks': 7}, 'confidence': 'HIGH'}, {'text': "Run the database seeds.\n*\n* @return void\n*/\npublic function run()\n{\nDB::table('users')->insert([\n'name' => Str::random(10),\n'email' => Str::random(10).'@gmail.com',\n'password' => Hash::make('password'),\n]);\n}\n}\nYou may type-hint any dependencies you need within the run method's signature. They will automatically be resolved via the Laravel service container.\nUsing Model Factories\nOf course, manually specifying the attributes for each model seed is cumbersome. Instead, you can use model factories to conveniently generate large amounts of database records. First, review the model factory documentation to learn how to define your factories. Once you have defined your factories, you may use the factory helper function to insert records into your database.\nFor example, let's create 50 users and attach a relationship to each user:\n/**\n* Run the database seeds.\n*\n* @return void\n*/\npublic function run()\n{\nfactory(App\\User::class, 50)->create()->each(function ($user) {\n$user->posts()->save(factory(App\\Post::class)->make());\n});\n}\nCalling Additional Seeders\nWithin the DatabaseSeeder class, you may use the call method to execute additional seed classes. Using the call method allows you to break up your database seeding into multiple files so that no single seeder class becomes overwhelmingly large. Pass the name of the seeder class you wish to run:\n/**\n* Run the database seeds.\n*\n* @return void\n*/\npublic function run()\n{\n$this->call([\nUserSeeder::class,\nPostSeeder::class,\nCommentSeeder::", 'similarity': 0.5927780270576477, 'metadata': {'total_chunks': 3, 'chunk_id': 1, 'source': 'seeder.txt'}, 'confidence': 'HIGH'}, {'text': ':\n$users = DB::table(\'users\')->distinct()->get();\nIf you already have a query builder instance and you wish to add a column to its existing select clause, you may use the addSelect method:\n$query = DB::table(\'users\')->select(\'name\');\n$users = $query->addSelect(\'age\')->get();\nRaw Expressions\nSometimes you may need to use a raw expression in a query. To create a raw expression, you may use the DB::raw method:\n$users = DB::table(\'users\')\n->select(DB::raw(\'count(*) as user_count, status\'))\n->where(\'status\', \'<>\', 1)\n->groupBy(\'status\')\n->get();\nRaw statements will be injected into the query as strings, so you should be extremely careful to not create SQL injection vulnerabilities.\nRaw Methods\nInstead of using DB::raw, you may also use the following methods to insert a raw expression into various parts of your query.\nselectRaw\nThe selectRaw method can be used in place of addSelect(DB::raw(...)). This method accepts an optional array of bindings as its second argument:\n$orders = DB::table(\'orders\')\n->selectRaw(\'price * ? as price_with_tax\', [1.0825])\n->get();\nwhereRaw / orWhereRaw\nThe whereRaw and orWhereRaw methods can be used to inject a raw where clause into your query. These methods accept an optional array of bindings as their second argument:\n$orders = DB::table(\'orders\')\n->whereRaw(\'price > IF(state = "TX", ?, 100)\', [200])\n->get();\nhavingRaw / orHavingRaw\nThe havingRaw and orHavingRaw methods may be used to set a raw', 'similarity': 0.580132007598877, 'metadata': {'chunk_id': 4, 'total_chunks': 21, 'source': 'queries.txt'}, 'confidence': 'HIGH'}]

In [ ]:
gen = generator.generate("Laravel database",filtered_chunks)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Filtered Chunks->[{'text': "<s> Database: Getting Started\nIntroduction\nConfiguration\nRead & Write Connections\nUsing Multiple Database Connections\nRunning Raw SQL Queries\nListening For Query Events\nDatabase Transactions\nIntroduction\nLaravel makes interacting with databases extremely simple across a variety of database backends using either raw SQL, the fluent query builder, and the Eloquent ORM. Currently, Laravel supports four databases:\nMySQL 5.6+ (Version Policy)\nPostgreSQL 9.4+ (Version Policy)\nSQLite 3.8.8+\nSQL Server 2017+ (Version Policy)\nConfiguration\nThe database configuration for your application is located at config/database.php. In this file you may define all of your database connections, as well as specify which connection should be used by default. Examples for most of the supported database systems are provided in this file.\nBy default, Laravel's sample environment configuration is ready to use with Laravel Homestead, which is a convenient virtual machine

In [ ]:
query = "Who is the prime minister of india?"
import torch
from transformers import pipeline

pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype=torch.bfloat16, device_map="auto")


messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot who always responds in the style of a pirate",
    },
    {"role": "user", "content": query},
]
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'top_p', 'top_k', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|system|>
You are a friendly chatbot who always responds in the style of a pirate</s>
<|user|>
Who is the prime minister of india?</s>
<|assistant|>
The current prime minister of India is Narendra Modi, who was sworn in on May 26, 2014.


In [ ]:
messages1= """<|system|>
You are a stock market expert.
-Think step by step
  -Max answer size 80
  -Answer honestly
  -No Noise
  -Short term analysis
  -Long term analysis
  -Answer using only the provided context if you don't know say I don't know
<|user|>
a retail investor
 Context:
  - Current price 190.52
  -Today's wipro price is approx 1.53% upside
  Question:Should I buy wipro stock?
 <|assistant|>"""

In [ ]:
prompt = pipe.tokenizer.apply_chat_template(messages1, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, do_sample=False, temperature=0.2, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|assistant|>

1. The first step to creating a successful social media strategy is to define your goals. What do you want to achieve with your social media presence? Is it to build brand awareness, generate leads, or increase sales?

2. Determine your target audience and their interests. Use social media analytics to understand who your followers are and what they are interested in.

3. Create a content strategy that aligns with your goals and audience. Determine what types of content you will post, when and how often, and how much time you will invest.

4. Use social media advertising to reach your target audience. Advertise on Facebook, Instagram, Twitter, LinkedIn, or YouTube. Use targeting options to ensure your ads are reaching the right people.

5. Measure your success by tracking metrics such as engagement, reach, and conversions. Use social media analytics to see what types of content are resonating with your audience and which platforms are driving the most traffic and leads.


<|system|>
You are a friendly chatbot who always responds in the style of a pirate</s>
<|user|>
Who is the prime minister of india?</s>
<|assistant|>
The current prime minister of India is Narendra Modi, who was sworn in on May 26, 2014.

In [ ]:
print(gen)

The Laravel database is very easy to use and provides a lot of flexibility. It supports various database backends including MySQL, PostgreSQL, SQLite, and SQL Server. To connect to a database, you simply need to define the connection in your `config/database.php` file. You can also specify which connection should be used by default. To add a new database connection, create a new `.env` file and add the following lines:

DB_CONNECTION=your_connection_name
DB_HOST=localhost
DB_PORT=3306
DB_DATABASE=your_database_name
DB_USERNAME=your_username
DB_PASSWORD=your_password

Then, you can use the `DB_CONNECTION` variable to connect to the desired database. For example, if you want to connect to a MySQL database named `my_database`, you would use the following code:

DB::
